##Chapter 6: Tools for Model Training and Experimenting/Project:
* Retrieval Augmented Generation Chatbot/CH6-01-Assembling ChatBot.py

In [0]:
%pip install -q mlflow==2.9.0 langchain==0.0.344 databricks-vectorsearch==0.22 databricks-sdk==0.12.0 mlflow==2.9
%restart_python

In [0]:
from mlflow.deployments import get_deploy_client
deploy_client = get_deploy_client("databricks")
endpoints = deploy_client.list_endpoints()
for endpoint in endpoints:
    print(endpoint['name'])

In [0]:
import os 
# url used to send the request to your model from the serverless endpoint
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get() 
#db_token = dbutils.secrets.get("mlaction", "rag_sp_token")
#db_host = dbutils.secrets.get("mlaction", "rag_sp_host")
os.environ['DATABRICKS_TOKEN'] = db_token
os.environ['DATABRICKS_HOST'] = host

In [0]:
from databricks.vector_search.client import VectorSearchClient
from langchain.vectorstores import DatabricksVectorSearch
from langchain.embeddings import DatabricksEmbeddings

catalog = "porya_catalog"
database_name = "default"

# NOTE: your question embedding model must match the one used in the chunk in the previous model 
embedding_model = DatabricksEmbeddings(endpoint="databricks-gte-large-en")
vsc_endpoint_name = "one-env-shared-endpoint-1" #"ml_action_vs"
index_name = f"{catalog}.{database_name}.docs_vsc_idx_cont"

def get_retriever(persist_dir: str = None):
    #Get the vector search index
    vsc = VectorSearchClient(workspace_url=os.environ['DATABRICKS_HOST'], personal_access_token=os.environ["DATABRICKS_TOKEN"])
    print("\n")
    vs_index = vsc.get_index(
        endpoint_name=vsc_endpoint_name,
        index_name=index_name)
    # Create the retriever
    vectorstore = DatabricksVectorSearch(
        vs_index, text_column="content", embedding=embedding_model)
    return vectorstore.as_retriever()

# test our retriever
vectorstore = get_retriever()
similar_documents = vectorstore.get_relevant_documents("What is GPT? ")
print(f"Relevant documents: {similar_documents[0]}")

In [0]:
import langchain
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatDatabricks

# Using Foundational Model from Databricks
# You could also use Llama70B or GPT4
chat_model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", 
                            max_tokens = 200)

TEMPLATE = """
You are an assistant for the AI Swat Team. You are answering questions related to the GenerativeAI and LLM's and how they impact humans life, labour, economic and financial impact. If the question is not related to one of these topics, kindly decline to answer. If you don't know the answer, just say that you don't know, don't try to make up an answer. Keep the answer as concise as possible. Use the following pieces of context to answer the question at the end:
{context}
Question: {question}
Answer:
"""
prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=get_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

# COMMAND ----------

langchain.debug = False #uncomment to see the chain details and the full prompt being sent
question = {"query": "Will AI impact work forces in the US ? "}
answer = chain.run(question)
print(answer,"\n")

question = {"query": "Can LLM's impact wages and how ? "}
answer = chain.run(question)
print(answer,"\n")

In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.tracking.client import MlflowClient

# Replacement for mlia_utils helpers
current_user = spark.sql("SELECT current_user()").first()[0]

def get_latest_model_version(model_name):
    client = MlflowClient()
    versions = client.search_model_versions(f"name='{model_name}'")
    return max([int(v.version) for v in versions])

# create experiment if does not exist 
experiment_path = f"/Users/{current_user}/mlaction_chatbot_rag"
mlflow.set_experiment(experiment_path) 

# COMMAND ----------

mlflow.set_registry_uri("databricks-uc")
model_name = f"{catalog}.{database_name}.mlaction_chatbot_model"

with mlflow.start_run(run_name="mlaction_chatbot_rag") as run:
    signature = infer_signature(question, answer)
    model_info = mlflow.langchain.log_model(
        chain,
        loader_fn=get_retriever,  # Load the retriever with DATABRICKS_TOKEN env as secret (for authentication).
        artifact_path="chain",
        registered_model_name=model_name,
        pip_requirements=[
            "mlflow==" + mlflow.__version__,
            "langchain==" + langchain.__version__,
            "databricks-vectorsearch",
        ],
        input_example=question,
        signature=signature
    )
    #------------------------
    import mlflow.models.utils
    mlflow.models.utils.add_libraries_to_model(
        f"models:/{model_name}/{get_latest_model_version(model_name)}"
    )